## 1. Data cleaning

In [ ]:
from datetime import datetime
import pandas as pd
import regex as re
from collections import Counter
import nltk

# nltk.download('stopwords')
# nltk.download('punkt')
# nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

#import nltk vader to sentiment analysis
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
#Install the relevant libraries for translation
!pip install wordcloud
!pip install langdetect googletrans==4.0.0-rc1 nltk
!pip install deep-translator

In [ ]:
from langdetect import detect
from deep_translator import GoogleTranslator

# Initialize translation
translator = GoogleTranslator(source='auto', target='en')

def translate_to_english(text):
    try:
        # Detect language
        lang = detect(text)
        # If it's not English, translate it
        if lang != 'en':
            text = translator.translate(text)
    except:
        pass  # If language detection or translation fails, keep the original text
    return text

# Test it
print(translate_to_english("Hola mundo"))  # Should translate "Hello world"
print(translate_to_english("Hello world"))  # Should keep as-is

In [ ]:
# Initialize stemmer and lemmatizer
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))
# Retain some important stop words
negations = {'not', 'no', 'nor'}
stop_words = stop_words - negations
# Add some stop words
custom_stopwords = {'yeah', 'okay', 'like', 'afternoon'}
stop_words = stop_words.union(custom_stopwords)


def clean_text(text):
    # Check if text is a string
    if isinstance(text, str):
        
        # Translate into English (if not English)
        text = translate_to_english(text)
        
        # Convert to lowercase
        text = text.lower()
        # Remove links
        text = re.sub(r'(https|http)(\S+)', '', text)
        # Remove emojis
        emoji_pattern = re.compile("["
                                   u"\U0001F600-\U0001F64F"  # emoticons
                                   u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                                   u"\U0001F680-\U0001F6FF"  # transport & map symbols
                                   u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                                   u"\U00002702-\U000027B0"  # Miscellaneous Symbols
                                   u"\U000024C2-\U0001F251"  # Enclosed Characters and Shapes
                                   "]+", flags=re.UNICODE)
        text = emoji_pattern.sub(r'', text)
        # Replace username
        text = re.sub(r'@(\w+)', '', text)
        # Remove punctuation
        text = re.sub(r'[^\w\s]|[\u0080-\uffff]', '', text)
        # Replace hashtag
        text = re.sub(r'#', '', text)
        # Replace retweets
        text = re.sub(r'rt', '', text)
        # Tokenize the text
        tokens = word_tokenize(text)
        # Remove stop words
        tokens = [token for token in tokens if token not in stop_words]
        # Perform stemming and lemmatization
        # stemmed_tokens = [stemmer.stem(token) for token in tokens]
        # lemmatized_tokens = [lemmatizer.lemmatize(token) for token in stemmed_tokens]
        # lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
        text = ' '.join(tokens)
        # Join tokens back into text
        # text = ' '.join(lemmatized_tokens)
        # Trim whitespace
        text = text.strip()
        # Remove extra spaces
        text = re.sub(r'\s+', ' ', text)        
        text = re.sub(r'  ', '', text)
    return text

In [ ]:
text = "你好"
print(translate_to_english(text),'||',clean_text(text))
text1 = "selamat petang" #こんにちは
print(translate_to_english(text1),'||',clean_text(text1))
text2 = "Check out this link: http://example.com #awesome RT @user1'"
print(translate_to_english(text2),'||',clean_text(text2))
text3 = "Pengalaman penerbangan ini tidak bagus"
print(translate_to_english(text3),'||',clean_text(text3))

## 2. Read data and apply cleaning

In [ ]:
import os
import pandas as pd

In [ ]:
!pip install tqdm

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# Register tqdm to support Pandas apply operation
tqdm.pandas()

def read_excel_and_clean(file_path, column_name):
    # Get the file extension
    file_ext = os.path.splitext(file_path)[1].lower()

    # If the file is CSV, read it and save as Excel
    if file_ext == '.csv':
        df = pd.read_csv(file_path, encoding='latin1')
        df = df.dropna(axis=1, how='all')  # Drop columns that are completely empty
        # Replace the extension with .xlsx
        file_path = file_path.rsplit('.', 1)[0] + '.xlsx'
        df.to_excel(file_path, index=False)
    else:
        df = pd.read_excel(file_path)

    # Store the original text in a new column
    df['original text'] = df[column_name]

    # Translate the text to English, displaying the progress bar
    print("Translating text to English...")
    df['translated text'] = df['original text'].progress_apply(translate_to_english)

    # Clean the translated text, displaying the progress bar
    print("Cleaning translated text...")
    df['cleansed data'] = df['translated text'].progress_apply(clean_text)

    # Remove rows where cleaned data is empty
    df = df[df['cleansed data'].str.strip().astype(bool)]

    return df


In [ ]:
# !pip install --upgrade openpyxl
# Example usage
file_path = "./Maxis_reviews.csv"
column_name = "content"

df_cleaned = read_excel_and_clean(file_path, column_name)
print(df_cleaned[column_name].head(5))
print("Data after translating:")
print(df_cleaned['translated text'].head(5))
print("Data after cleaning:")
print(df_cleaned['cleansed data'].head(5))

## 3. Sentiment analysis

In [ ]:
#import nltk vader to sentiment analysis
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
def analyze_sentiment(df, column_name):
    # Create sentiment analyzer object
    analyzer = SentimentIntensityAnalyzer()
    
    # Sentiment analysis function
    def analyze_single_text(text):
        text = str(text)  # Ensure input is string
        sentiment_score = analyzer.polarity_scores(text)
        compound_score = sentiment_score['compound']
        if compound_score > 0.05:
            return 'Positive'
        elif compound_score < -0.05:
            return 'Negative'
        else:
            return 'Neutral'

    # Perform sentiment analysis on specific column and add results to a new column
    df['Sentiment'] = df[column_name].apply(analyze_single_text)
    df['Compound'] = df[column_name].apply(lambda x: analyzer.polarity_scores(str(x))['compound'])  # optional


    return df


In [ ]:
df_cleaned_sentiment = analyze_sentiment(df_cleaned, 'cleansed data')
print(df_cleaned_sentiment['Sentiment'].head(5))

## 4. Save file

In [ ]:
def write_to_excel(df, output_file):
    # Remove fully empty columns before saving
    df = df.dropna(axis=1, how='all')
    # Write the DataFrame with sentiment analysis results to Excel
    df.to_excel(output_file, index=False)

In [ ]:
output_file = 'output_Trax_Maxis_reviews.xlsx'
write_to_excel(df_cleaned_sentiment, output_file)

In [ ]:
# Obtain the location of the file
import os
os.path.abspath('.')

## 5. A part of the data statistics and visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import make_interp_spline

def plot_selected_sentiment_trend_yearly(df, selected_sentiment='Positive', sentiment_column='Sentiment', save_path=None):
    """
    Plot smoothed yearly trend for a selected sentiment.

    Parameters:
        df: DataFrame with a date and sentiment column
        selected_sentiment: 'Positive', 'Negative', or 'Neutral'
        sentiment_column: name of the sentiment column
        save_path: optional path to save figure at 300 dpi
    """
    df = df.copy()

    # Auto-detect and parse the date column
    if 'Review Date' in df.columns:
        df['ParsedDate'] = pd.to_datetime(
            df['Review Date'].str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True),
            errors='coerce'
        )
    elif 'at' in df.columns:
        df['ParsedDate'] = pd.to_datetime(df['at'], errors='coerce')
    else:
        raise ValueError("Expected 'Review Date' or 'at' column for date information.")

    df = df.dropna(subset=['ParsedDate', sentiment_column])
    df['Year'] = df['ParsedDate'].dt.year

    # Filter by selected sentiment
    df_filtered = df[df[sentiment_column] == selected_sentiment]

    # Aggregate by year
    yearly_counts = df_filtered.groupby('Year').size().sort_index()

    # Prepare data
    x = np.arange(len(yearly_counts))
    y = yearly_counts.values
    years = yearly_counts.index

    plt.figure(figsize=(16, 10))

    if len(x) >= 4:
        x_smooth = np.linspace(x.min(), x.max(), 300)
        spline = make_interp_spline(x, y, k=3)
        y_smooth = spline(x_smooth)
        plt.plot(x_smooth, y_smooth, label=selected_sentiment, color='teal')
    else:
        plt.plot(x, y, label=selected_sentiment, color='teal')

    plt.xticks(
        ticks=np.arange(len(years)),
        labels=[str(year) for year in years],
        rotation=45
    )
    plt.xlabel("Year", fontsize=22)
    plt.ylabel("Number of Reviews", fontsize=22)
    plt.title(f"Yearly Trend of {selected_sentiment} Sentiment", fontsize=26)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.legend(fontsize=22)
    plt.tight_layout()

    plt.tight_layout()


    plt.savefig(f"Yearly Trend of {selected_sentiment} Sentiment_yearly_Maxis_reviews.jpg", dpi=300)

    plt.show()


In [ ]:
plot_selected_sentiment_trend_yearly(df_cleaned, selected_sentiment='Positive')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import make_interp_spline

def plot_selected_sentiment_trend_monthly(df, selected_sentiment='Neutral', sentiment_column='Sentiment', save_path=None):
    """
    Plot smoothed monthly trend for a selected sentiment.

    Parameters:
        df: DataFrame containing sentiment and a time column
        selected_sentiment: 'Positive', 'Negative', or 'Neutral'
        sentiment_column: name of the sentiment column
        save_path: optional path to save the figure at 300 dpi
    """
    df = df.copy()

    # Auto-detect and parse the date
    if 'Review Date' in df.columns:
        df['ParsedDate'] = pd.to_datetime(
            df['Review Date'].str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True),
            errors='coerce'
        )
    elif 'at' in df.columns:
        df['ParsedDate'] = pd.to_datetime(df['at'], errors='coerce')
    else:
        raise ValueError("No supported time column found. Expected 'Review Date' or 'at'.")

    df = df.dropna(subset=['ParsedDate', sentiment_column])
    df['Month'] = df['ParsedDate'].dt.to_period('M').dt.to_timestamp()

    # Filter by selected sentiment
    df_filtered = df[df[sentiment_column] == selected_sentiment]

    # Aggregate by month
    monthly_counts = df_filtered.groupby('Month').size().sort_index()

    # Prepare data
    x = np.arange(len(monthly_counts))
    y = monthly_counts.values
    months = monthly_counts.index

    plt.figure(figsize=(16, 10))

    if len(x) >= 4:
        x_smooth = np.linspace(x.min(), x.max(), 300)
        spline = make_interp_spline(x, y, k=3)
        y_smooth = spline(x_smooth)
        plt.plot(x_smooth, y_smooth, label=selected_sentiment, color='steelblue')
    else:
        plt.plot(x, y, label=selected_sentiment, color='steelblue')

    plt.xticks(
        ticks=np.arange(len(months)),
        labels=[m.strftime("%Y-%m") for m in months],
        rotation=45
    )
    plt.xlabel("Month", fontsize=22)
    plt.ylabel("Number of Reviews", fontsize=22)
    plt.title(f"Monthly Trend of {selected_sentiment} Sentiment", fontsize=26)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.legend(fontsize=22)
    plt.tight_layout()

    plt.savefig(f"Monthly Trend of {selected_sentiment} Sentiment_Monthly_Maxis_reviews.jpg", dpi=300)

    plt.show()


In [ ]:
plot_selected_sentiment_trend_monthly(df_cleaned, selected_sentiment='Negative')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import make_interp_spline

def plot_selected_sentiment_trend(df, selected_sentiment='Positive', sentiment_column='Sentiment', save_path=None):
    """
    Plot smoothed trend over time for a selected sentiment.

    Parameters:
        df: DataFrame containing sentiment and a time column
        selected_sentiment: 'Positive', 'Negative', or 'Neutral'
        sentiment_column: name of the sentiment column
        save_path: optional path to save the figure at 300 dpi
    """
    df = df.copy()

    # Auto-detect and parse the date
    if 'Review Date' in df.columns:
        df['ParsedDate'] = pd.to_datetime(
            df['Review Date'].str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True),
            errors='coerce'
        )
    elif 'at' in df.columns:
        df['ParsedDate'] = pd.to_datetime(df['at'], errors='coerce')
    else:
        raise ValueError("No supported time column found. Expected 'Review Date' or 'at'.")

    df = df.dropna(subset=['ParsedDate', sentiment_column])
    df['Date'] = df['ParsedDate'].dt.date

    # Filter to selected sentiment
    df_filtered = df[df[sentiment_column] == selected_sentiment]

    # Count sentiment per date
    sentiment_counts = df_filtered.groupby('Date').size().sort_index()

    # Prepare data
    x = np.arange(len(sentiment_counts))
    y = sentiment_counts.values
    dates = sentiment_counts.index

    plt.figure(figsize=(16, 10))

    if len(x) >= 4:
        x_smooth = np.linspace(x.min(), x.max(), 300)
        spline = make_interp_spline(x, y, k=3)
        y_smooth = spline(x_smooth)
        plt.plot(x_smooth, y_smooth, label=selected_sentiment, color='steelblue')
    else:
        plt.plot(x, y, label=selected_sentiment, color='steelblue')

    plt.xticks(
        ticks=np.arange(len(dates)),
        labels=[d.strftime("%Y-%m-%d") for d in dates],
        rotation=45
    )
    plt.xlabel("Date")
    plt.ylabel("Number of Reviews")
    plt.title(f"Daily Trend of {selected_sentiment} Sentiment Over Time")
    plt.legend()
    plt.tight_layout()

    plt.savefig(f"Daily Trend of {selected_sentiment} Sentiment_Monthly Maxis_reviews.jpg", dpi=300)

    plt.show()


In [ ]:
plot_selected_sentiment_trend(df_cleaned, selected_sentiment='Positive')


In [ ]:
from collections import Counter
from nltk.corpus import stopwords

# def calculate_word_freq_by_sentiment(df, text_column, sentiment_column='Sentiment'):
#     # Dictionary to hold word frequency per sentiment
#     sentiment_word_freq = {}

#     # Sentiment-specific word frequencies
#     for sentiment in ['Positive', 'Negative', 'Neutral']:
#         filtered_df = df[df[sentiment_column] == sentiment]
#         cleaned_text = ' '.join(filtered_df[text_column].dropna().values)
#         words = cleaned_text.split()
#         word_freq = Counter(words)
#         sentiment_word_freq[sentiment] = word_freq

#     # Overall word frequency (all sentiments)
#     all_text = ' '.join(df[text_column].dropna().values)
#     all_words = all_text.split()
#     sentiment_word_freq['All'] = Counter(all_words)

#     return sentiment_word_freq


def calculate_word_freq_by_sentiment(df, text_column, sentiment_column='Sentiment'):
    # Load English stopwords
    stop_words = set(stopwords.words('english'))
    # Retain some important stop words
    # negations = {'not', 'no', 'nor'}
    # stop_words = stop_words - negations
    # # Add some stop words
    # custom_stopwords = {'yeah', 'okay', 'like', 'afternoon'}
    # stop_words = stop_words.union(custom_stopwords)

    # Dictionary to hold word frequency per sentiment
    sentiment_word_freq = {}

    # Sentiment-specific word frequencies
    for sentiment in ['Positive', 'Negative', 'Neutral']:
        filtered_df = df[df[sentiment_column] == sentiment]
        cleaned_text = ' '.join(filtered_df[text_column].dropna().values)
        words = cleaned_text.split()
        filtered_words = [word for word in words if word.lower() not in stop_words]
        word_freq = Counter(filtered_words)
        sentiment_word_freq[sentiment] = word_freq

    # Overall word frequency (all sentiments)
    all_text = ' '.join(df[text_column].dropna().values)
    all_words = all_text.split()
    all_filtered_words = [word for word in all_words if word.lower() not in stop_words]
    sentiment_word_freq['All'] = Counter(all_filtered_words)

    return sentiment_word_freq


In [ ]:
# # Calculate word frequencies after removing stopwords
# word_freq = calculate_word_freq(df_cleaned_sentiment, 'cleansed data')

# # Print the top 20 words and their frequencies
# print("Top 20 words and their frequencies after cleaning and removing stopwords:")
# for word, freq in word_freq.most_common(20):
#     print(word, freq)
word_frequencies = calculate_word_freq_by_sentiment(df_cleaned_sentiment, 'cleansed data')
all_word_freq = word_frequencies['All'].most_common()
pos_word_freq = word_frequencies['Positive'].most_common()
neg_word_freq = word_frequencies['Negative'].most_common()
neu_word_freq = word_frequencies['Neutral'].most_common()

print('All: ', word_frequencies['All'].most_common(10))        # The first 10 words of all emotions
print('Positive: ',word_frequencies['Positive'].most_common(10))   # The first 10 words of positive
print('Negative: ',word_frequencies['Negative'].most_common(10))   # The first 10 words of Negative
print('Neutral: ',word_frequencies['Neutral'].most_common(10))   # The first 10 words of Neutral


In [ ]:
import matplotlib.pyplot as plt
# !pip install wordcloud
from wordcloud import WordCloud

In [ ]:
# Plot a bar chart
def plot_word_freq(word_freq_dict, sentiment='All', top_n=20):
    if sentiment not in word_freq_dict:
        print(f"Sentiment '{sentiment}' not found. Choose from {list(word_freq_dict.keys())}")
        return
    
    freq_data = word_freq_dict[sentiment].most_common(top_n)
    
    if not freq_data:
        print(f"No words to display for sentiment: {sentiment}")
        return

    words, freqs = zip(*freq_data)

    plt.figure(figsize=(16, 10))
    plt.bar(words, freqs, color='skyblue')
    plt.title(f"Top {top_n} Words in {sentiment} Sentiment", fontsize=26)
    plt.xlabel("Words", fontsize=22)
    plt.ylabel("Frequency", fontsize=22)
    plt.xticks(rotation=30, ha='right', fontsize=18)
    plt.yticks(fontsize=18)
    plt.tight_layout()
    plt.savefig(f"Top {top_n} Words in {sentiment} Sentiment_Maxis_reviews_Positive.jpg", dpi=300)
    plt.show()

plot_word_freq(word_frequencies, sentiment='Positive', top_n=10)

In [ ]:
# Plot a bar chart
def plot_word_freq(word_freq_dict, sentiment='All', top_n=20):
    if sentiment not in word_freq_dict:
        print(f"Sentiment '{sentiment}' not found. Choose from {list(word_freq_dict.keys())}")
        return
    
    freq_data = word_freq_dict[sentiment].most_common(top_n)
    
    if not freq_data:
        print(f"No words to display for sentiment: {sentiment}")
        return

    words, freqs = zip(*freq_data)

    plt.figure(figsize=(16, 10))
    plt.bar(words, freqs, color='skyblue')
    plt.title(f"Top {top_n} Words in {sentiment} Sentiment", fontsize=26)
    plt.xlabel("Words", fontsize=22)
    plt.ylabel("Frequency", fontsize=22)
    plt.xticks(rotation=30, ha='right', fontsize=18)
    plt.yticks(fontsize=18)
    plt.tight_layout()
    plt.savefig(f"Top {top_n} Words in {sentiment} Sentiment_Maxis_reviews-Negative.jpg", dpi=300)
    plt.show()

plot_word_freq(word_frequencies, sentiment='Negative', top_n=10)

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def plot_wordcloud(word_freq_dict, sentiment='All'):

    if sentiment not in word_freq_dict:
        print(f"Sentiment '{sentiment}' not found. Choose from {list(word_freq_dict.keys())}")
        return

    freq = word_freq_dict[sentiment]
    if not freq:
        print(f"No words to display for sentiment: {sentiment}")
        return

    wc = WordCloud(
        width=800, 
        height=400, 
        background_color='white',
        colormap='viridis'  # change color: 'viridis', 'plasma', 'inferno', 'magma', 'cool', 'spring', 'summer', 'autumn', 'winter', 'cividis'
     ).generate_from_frequencies(freq)

    plt.figure(figsize=(10, 6))
    plt.title(f"Words Cloud for {sentiment} Sentiment", fontsize=26)
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    # plt.title(f'Word Cloud for {sentiment} Sentiment')
    plt.tight_layout()
    plt.savefig(f"Word Cloud for {sentiment} Maxis_reviews Sentiment_Negative.jpg", dpi=300)
    plt.show()


plot_wordcloud(word_frequencies, sentiment='Negative')


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def plot_wordcloud(word_freq_dict, sentiment='All'):

    if sentiment not in word_freq_dict:
        print(f"Sentiment '{sentiment}' not found. Choose from {list(word_freq_dict.keys())}")
        return

    freq = word_freq_dict[sentiment]
    if not freq:
        print(f"No words to display for sentiment: {sentiment}")
        return

    wc = WordCloud(
        width=800, 
        height=400, 
        background_color='white',
        colormap='viridis'  # change color: 'viridis', 'plasma', 'inferno', 'magma', 'cool', 'spring', 'summer', 'autumn', 'winter', 'cividis'
     ).generate_from_frequencies(freq)

    plt.figure(figsize=(10, 6))
    plt.title(f"Words Cloud for {sentiment} Sentiment", fontsize=26)
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    # plt.title(f'Word Cloud for {sentiment} Sentiment')
    plt.tight_layout()
    plt.savefig(f"Word Cloud for {sentiment} Maxis_reviews_Sentiment_Positive.jpg", dpi=300)
    plt.show()


plot_wordcloud(word_frequencies, sentiment='Positive')

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def plot_wordcloud(word_freq_dict, sentiment='All'):

    if sentiment not in word_freq_dict:
        print(f"Sentiment '{sentiment}' not found. Choose from {list(word_freq_dict.keys())}")
        return

    freq = word_freq_dict[sentiment]
    if not freq:
        print(f"No words to display for sentiment: {sentiment}")
        return

    wc = WordCloud(
        width=800, 
        height=400, 
        background_color='white',
        colormap='viridis'  # change color: 'viridis', 'plasma', 'inferno', 'magma', 'cool', 'spring', 'summer', 'autumn', 'winter', 'cividis'
     ).generate_from_frequencies(freq)

    plt.figure(figsize=(10, 6))
    plt.title(f"Words Cloud for {sentiment} Sentiment", fontsize=26)
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    # plt.title(f'Word Cloud for {sentiment} Sentiment')
    plt.tight_layout()
    plt.savefig(f"Word Cloud for {sentiment} Maxis_reviews Sentiment_Neutral.jpg", dpi=300)
    plt.show()


plot_wordcloud(word_frequencies, sentiment='Neutral')

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

def plot_wordcloud(word_freq_dict, sentiment='All'):

    if sentiment not in word_freq_dict:
        print(f"Sentiment '{sentiment}' not found. Choose from {list(word_freq_dict.keys())}")
        return

    freq = word_freq_dict[sentiment]
    if not freq:
        print(f"No words to display for sentiment: {sentiment}")
        return

    wc = WordCloud(
        width=800, 
        height=400, 
        background_color='white',
        colormap='viridis'  # change color: 'viridis', 'plasma', 'inferno', 'magma', 'cool', 'spring', 'summer', 'autumn', 'winter', 'cividis'
     ).generate_from_frequencies(freq)

    plt.figure(figsize=(10, 6))
    plt.title(f"Words Cloud for {sentiment} Sentiment", fontsize=26)
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    # plt.title(f'Word Cloud for {sentiment} Sentiment')
    plt.tight_layout()
    plt.savefig(f"Word Cloud for {sentiment} Maxis_reviews Sentiment_all.jpg", dpi=300)
    plt.show()


plot_wordcloud(word_frequencies, sentiment='All')

In [ ]:
import matplotlib.pyplot as plt

def generate_pie_chart(df):
    sentiment_counts = df['Sentiment'].value_counts()
    colors = ['#e5f5e0','#a1d99b','#31a354','#238b45'] # You can adjust the color according to the number of categories
    plt.figure(figsize=(12, 10))
    wedges, texts, autotexts = plt.pie(
        sentiment_counts,
        labels=sentiment_counts.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        textprops={'fontsize': 16}  #Set the font size of the labels and percentages
    )
    plt.setp(autotexts, size=14, weight="bold")  # The percentage text is bolded
    plt.axis('equal')  # Keep the pie chart round
    # plt.title('Sentiment Distribution', fontsize=20)
    plt.savefig('Pie_chart_Maxis_reviews.jpg', dpi=300)
    plt.show()

generate_pie_chart(df_cleaned_sentiment)
